<h1 align="center">PSL Week - Topological Data Analysis</h1>
<h1 align="center">Exam: Data project</h1>


# Evaluation

Your task is to study the provided data and design a supervised classification procedure to distinguish between the different classes (ie persons) and apply your classifier to predict the labels of the unlabeled data provided as test set.

**By groups of two**, you will present your work in a **10–15 minute oral presentation**, using both slides or the blackboard, followed by **5–10 minutes of questions**. Your presentation should clearly demonstrate that you understand the concepts, methods, and tools you used in your analysis. In particular, any filtration, machine learning method, or other technique you rely on must be briefly introduced and explained during your talk. Interpretation of visual outputs (plots, embeddings, diagrams, etc.) is strongly encouraged and will be valued.

During the presentation, it is expected that you show the result of your evaluation as a list of labels and the corresponding count. Something like

```plaintext
List of labels: ['A', 'B', 'A', 'A', 'C', 'A', 'C', ...]
Total number of class A: 8
Total number of class B: 5
Total number of class A: 9
```


The general pipeline of a TDA machine learning project is as follows

- Choose a simplicial complex, filtration and maximal dimension, and extract the diagrams up to the chosen maximal dimension. 
- Apply a vectorization method to extract features from the diagrams
- Train a classification model on the vectorized diagrams

Each step is important and needs to be investigated with respect to your dataset. You may want to choose to focus on one of the blocks or all of them, but keep in mind that your analysis of what works and what doesn't will be valued.

# Data

The walking motion of three individuals, A, B, and C, was recorded using the accelerometer of a smartphone placed in their pocket, producing three multivariate time series in $\mathbb{R}^3$. Each time series corresponds to the three acceleration components measured in the sensor’s coordinate system. Note that, since the phone may have been oriented differently for each person, these series cannot be compared component-wise.
Using a sliding window, each series was segmented into $N_A = N_B = N_C = 80$ time series of $p = 200$ consecutive points.

The exam consist in the analysis and evaluation of the data available [here](https://www.math.ens.psl.eu/~eaamari/teaching/2025-2026/PSL_Week/psl_week_acc_data.zip)
Extract the zip file in the directory of your notebook. You should see two files.

- `psl_week_acc_train_data.pkl` contains a numpy array with all the accelerometer data and the corresponding label array. Labels are given as `A`, `B` or `C`.

- `psl_week_acc_eval_data.pkl` contains a numpy array with $22$ unlabeled accelerometer data for the evaluation. These are the ones that you should actually classify without knowing the right answer! **During your presentation, present how many of each class $\hat{n}_A,\hat{n}_B,\hat{n}_C$ you found.**


# This notebook
The accompanying notebook aims at presenting the data, and offers one possible roadmap for a topological analysis of this dataset with guiding questions. It provides methods for exploring and visualizing, analyzing and transforming the data.

Please use it as a guideline, but make sure to come up with your own notebook showcasing your reasoning, analysis and conclusions. Keep in mind that relevant plots will greatly help explaining your approach, from data exploration, through method selection and modeling, to interpretation of results, and will add a lot of value to your presentation.


# Available tools

Here we provide a non exhaustive list of mehods that can be used for the different steps.
[Here](https://gudhi.inria.fr/python/latest/) is a link to the full documentation of Gudhi

## Complexes

Automatic ways to create filtration from point clouds

- `gd.AlphaComplex` (see [documentation](https://gudhi.inria.fr/python/3.10.0/alpha_complex_ref.html))
- `gd.RipsComplex` (see [documentation](https://gudhi.inria.fr/python/3.10.0/rips_complex_ref.html))

## Diagram distances

Can be used to observe how different classes are distant in the diagram space

- Botleneck distance (see [documenation](https://gudhi.inria.fr/python/latest/bottleneck_distance_user.html))
- Wasserstein distance (see [documenation](https://gudhi.inria.fr/python/latest/wasserstein_distance_user.html))

(a helper function `compute_distance_matrix` is provided further in the notebook)

## Visualization methods

From distance matrices or kernel matrices, some methods can be used to visualize how the different elements of the dataset occupy the feature space regarding the provided distance or kernel. You can find some examples in the second practical session and in this notebook.

- `sklearn.decomposition.PCA` is useful on explicit maps (see [documenation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html))
- `sklearn.decomposition.KernelPCA` is useful on kernel representations (see [documenation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.KernelPCA.html))
- `sklearn.manifold.MDS` can be applied on any distance matrix (see [documenation](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.MDS.html))

## Vectorization methods

More details on each vectorization method can be found in the second practical session

- Landscapes
- Silouhette
- Persistence Image
- Kernels (Persistence Scale Space, Persistence Fisher, Scaled Wasserstein)

## Classifiers

Those are generic scikit classifiers commonly used in machine learning.

- `sklearn.ensemble.RandomForestClassifier` (see [documenation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html))
- `sklearn.svm.SVC` (see [documenation](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html))
- `sklearn.ensemble.RandomForestClassifier` (see [documenation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html))

# Environment setup

Make sure you provide the correct path to the train and eval set and verify it by running the following code

In [ ]:
import numpy as np
import pickle as pickle
import gudhi as gd
from gudhi.hera import wasserstein_distance
import gudhi.representations as gdr
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm
import plotly
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from sklearn import manifold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.cluster import KMeans
import multiprocessing as mp
from typing import Literal
from dataclasses import dataclass, field

with open("psl_week_acc_train_data.pkl", "rb") as f:
    data, train_labels_np = pickle.load(f)
    train_labels = [label.decode('utf-8') for label in train_labels_np]

with open("psl_week_acc_eval_data.pkl", "rb") as f:
    eval_data = pickle.load(f)

print(data.shape, len(train_labels), eval_data.shape)
data_A = data[np.where(train_labels_np == b'A')]
data_B = data[np.where(train_labels_np == b'B')]
data_C = data[np.where(train_labels_np == b'C')]

## Helper functions
Here are some helper functions that you can use throughout the project
- `plot_3d_point_series` for visualizing any sequence of 3D points
- `compute_distance_matrix` which speeds up the computation of distance matrices based on a list of diagrams for Bottleneck and Wasserstein distances
- `plot_distance_matrices` that displays a detailed version of a matrix distance (**useful only when the size of the matrix is $<20$**). For bigger matrices use regular `plt.imshow`.
- `plot_mds` that displays spatially points based on a provided distance matrix. Usful to get a sense of how well classes are separated based on a given distance. Works in 2D and 3D.
- `create_interactive_vectorization_plot` which allows you to play with hyperparameters of vectorization methods. It comes with a class called `ParameterSlider` which discribes the range and meaning of the parameter. (An example is given at the end)

In [ ]:
def plot_3d_point_series(series_dict, title="3D Point Series", show_lines=False):
    """
    Plots multiple 3D point series stored in a dictionary of NumPy arrays.

    Parameters
    ----------
    series_dict : dict[str, np.ndarray]
        Dictionary mapping series names to Nx3 numpy arrays of 3D points.
    title : str
        Plot title.
    show_lines : bool
        Whether to connect points with lines.
    """
    fig = go.Figure()

    for name, arr in series_dict.items():
        if not isinstance(arr, np.ndarray):
            raise TypeError(f"Value for '{name}' must be a NumPy array.")
        if arr.ndim != 2 or arr.shape[1] != 3:
            raise ValueError(f"Array for '{name}' must be of shape (N, 3).")

        fig.add_trace(
            go.Scatter3d(
                x=arr[:, 0],
                y=arr[:, 1],
                z=arr[:, 2],
                mode='lines+markers' if show_lines else 'markers',
                marker=dict(size=2, opacity=0.8),
                line=dict(width=2),
                name=name  # Legend entry
            )
        )

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z"
        ),
        legend=dict(itemsizing="constant")
    )

    fig.show()

def plot_distance_matrices(distance_matrices_dict, diag_names=None):
    """
    Plot distance matrices side by side with proper labels.
    Readable up to matrices of size 20x20.
    
    Parameters:
    -----------
    distance_matrices_dict : dict
        Dictionary where keys are distance names and values are distance matrices
    diag_names : list, optional
        List of names for the diagrams. If None, uses default names.
    """
    
    # Convert to list for easier handling
    matrix_names = list(distance_matrices_dict.keys())
    matrices = list(distance_matrices_dict.values())
    
    # Default diagram names if not provided
    if diag_names is None:
        matrix_size = matrices[0].shape[0]
        diag_names = [str(i) for i in range(matrix_size)]
    
    # Process matrices in batches of 2
    num_matrices = len(matrices)
    num_batches = (num_matrices + 1) // 2  # Ceiling division
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * 2
        end_idx = min(start_idx + 2, num_matrices)
        batch_matrices = matrices[start_idx:end_idx]
        batch_names = matrix_names[start_idx:end_idx]
        
        # Determine subplot layout
        num_plots = len(batch_matrices)
        if num_plots == 1:
            fig, ax = plt.subplots(1, 1, figsize=(8, 6))
            axes = [ax]
        else:
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Plot each matrix in the current batch
        for i, (matrix, name) in enumerate(zip(batch_matrices, batch_names)):
            ax = axes[i] if num_plots > 1 else axes[0]
            
            # Create heatmap
            im = ax.imshow(matrix, cmap='viridis', interpolation='nearest')
            ax.set_title(f'{name} Matrix', fontsize=14, fontweight='bold')
            ax.set_xlabel('Diagrams', fontsize=12)
            ax.set_ylabel('Diagrams', fontsize=12)
            
            # Set ticks and labels
            ax.set_xticks(range(len(diag_names)))
            ax.set_yticks(range(len(diag_names)))
            ax.set_xticklabels(diag_names, rotation=45, ha='right')
            ax.set_yticklabels(diag_names)
            
            # Add colorbar
            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label('Distance', fontsize=11)
            
            # Add text annotations
            for row in range(len(diag_names)):
                for col in range(len(diag_names)):
                    text_color = "white" if matrix[row, col] > matrix.max()/2 else "black"
                    ax.text(col, row, f'{matrix[row, col]:.3f}',
                           ha="center", va="center", color=text_color, fontsize=9)
        
        # Hide unused subplot if only one matrix in this batch
        if num_plots == 1 and len(axes) > 1:
            axes[1].set_visible(False)
        
        plt.tight_layout()
        plt.show()

def compute_distance_matrix(diagram_list, distance_type: Literal['bottleneck', 'wasserstein']):
    """
    Compute pairwise distance matrix between persistence diagrams using multiprocessing.
    
    This function computes the pairwise distances between all persistence diagrams in the 
    input list using either bottleneck or Wasserstein distance. The computation is 
    parallelized using multiprocessing for efficiency, and progress is reported during 
    execution.
    
    Parameters
    ----------
    diagram_list : list
        List of persistence diagrams. Each diagram should be in a format compatible 
        with gudhi distance functions (e.g., numpy array of shape (n_points, 2) or 
        gudhi persistence diagram format).
    distance_type : Literal['bottleneck', 'wasserstein']
        Type of distance metric to compute between diagrams. Currently only 
        'bottleneck' is implemented in the function body.
        - 'bottleneck': Computes bottleneck distance using gudhi.bottleneck_distance
        - 'wasserstein': Not yet implemented (function body needs modification)
    
    Returns
    -------
    np.ndarray
        Symmetric distance matrix of shape (nb_data, nb_data) where nb_data is the 
        length of diagram_list. Entry (i,j) contains the distance between 
        diagram_list[i] and diagram_list[j]. The diagonal contains zeros.
    
    """
    if distance_type not in ['bottleneck', 'wasserstein']:
        raise ValueError("distance_type must be either 'bottleneck' or 'wasserstein'")
    
    nb_data = len(diagram_list)
    B= np.zeros((nb_data,nb_data))

    with mp.Pool() as pool:
        indexes = [(i, j, diagram_list[i], diagram_list[j]) for i in range(nb_data) for j in range(i)]
        next_prop = 0.1
        count = 0
        tot = len(indexes)
        for (i, j, b0) in pool.imap(_compute_bottleneck_distance if distance_type == 'bottleneck' else _compute_wasserstein_distance, indexes, chunksize=5):
            B[i,j] = b0
            B[j,i] = b0
            count += 1
            if count/tot > next_prop:
                print(f"{100*count/tot:.1f}% processing done")
                next_prop += 0.1
    
    return B

def _compute_bottleneck_distance(args):
    i, j, diag_0, diag_1= args
    b0 = gd.bottleneck_distance(diag_0, diag_1)
    return (i, j, b0)

def _compute_wasserstein_distance(args):
    i, j, diag_0, diag_1= args
    w0 = wasserstein_distance(diag_0, diag_1, order=2, internal_p=2)
    return (i, j, w0)

def plot_mds(distance_matrix, labels, visualization_dim: Literal[2, 3], title="MDS Visualization", 
                color_map=None, max_iter=3000, eps=1e-9):
    """
    Create a 2D or 3D visualization of a distance matrix using Multidimensional Scaling (MDS).
    
    Parameters
    ----------
    distance_matrix : np.ndarray
        Symmetric distance matrix of shape (n_samples, n_samples)
    labels : list or np.ndarray
        List of labels corresponding to each sample. Must have same length as distance_matrix dimensions.
    visualization_dim : Literal[2, 3]
        Dimension of the visualization (2D or 3D)
    title : str, optional
        Title for the 3D plot (default: "MDS 3D Visualization")
    color_map : dict, optional
        Dictionary mapping label values to colors. If None, uses default colors.
    max_iter : int, optional
        Maximum number of iterations for MDS (default: 3000)
    eps : float, optional
        Relative tolerance for convergence (default: 1e-9)
    
    Returns
    -------
    pos : np.ndarray
        The 3D embedding coordinates of shape (n_samples, 3)
    
    Examples
    --------
    >>> labels = ['A'] * 80 + ['B'] * 80 + ['C'] * 80
    >>> pos = plot_mds_3d(B1, labels, title="Persistence Diagrams MDS")
    """
    
    # Validate inputs
    if distance_matrix.shape[0] != distance_matrix.shape[1]:
        raise ValueError("Distance matrix must be square")
    if len(labels) != distance_matrix.shape[0]:
        raise ValueError("Labels length must match distance matrix dimensions")
    if visualization_dim not in [2, 3]:
        raise ValueError("visualization_dim must be either 2 or 3")
    
    # Get unique labels and create color mapping
    unique_labels = list(set(labels))
    
    if color_map is None:
        default_colors = ["red", "green", "blue", "purple", "orange", "brown", 
                         "pink", "gray", "olive", "cyan"]
        color_map = {label: default_colors[i % len(default_colors)] 
                    for i, label in enumerate(unique_labels)}
    
    # Apply MDS transformation
    mds = manifold.MDS(n_components=visualization_dim, max_iter=max_iter, eps=eps,
                       dissimilarity="precomputed", n_jobs=1)
    pos = mds.fit(distance_matrix).embedding_
    
    # Create data dictionary for plotting
    plot_data = {}
    for label in unique_labels:
        # Get indices for this label
        label_indices = [i for i, l in enumerate(labels) if l == label]
        plot_data[str(label)] = pos[label_indices]
    
    # Create 3D plot
    if visualization_dim == 3:
        plot_3d_point_series(plot_data, title=title, show_lines=False)
    elif visualization_dim == 2:
        plt.figure()
        for label, points in plot_data.items():
            plt.scatter(points[:, 0], points[:, 1], label=label, color=color_map.get(label, None))
        plt.title(title)
        plt.legend()
        plt.show()
    else:
        raise ValueError("visualization_dim must be either 2 or 3")
    
    return pos

@dataclass
class ParameterSlider:
    """
    Class for configuring a slider in an interactive plot.
    Validation performed in __post_init__.
    """
    name: str
    min_value: float
    max_value: float
    num_steps: int = field(default=10)
    log_scale: bool = field(default=False)
    is_integer: bool = field(default=False)

    def __post_init__(self):
        # name
        if not isinstance(self.name, str) or not self.name.strip():
            raise ValueError("ParameterSlider.name must be a non-empty string.")

        # numeric bounds
        try:
            self.min_value = float(self.min_value)
            self.max_value = float(self.max_value)
        except (TypeError, ValueError):
            raise ValueError("ParameterSlider.min_value and .max_value must be numbers.")

        if self.min_value >= self.max_value:
            raise ValueError("ParameterSlider.min_value must be less than max_value.")

        # num_steps
        if not isinstance(self.num_steps, int):
            try:
                self.num_steps = int(self.num_steps)
            except Exception:
                raise ValueError("ParameterSlider.num_steps must be an integer >= 2.")
        if self.num_steps < 2:
            raise ValueError("ParameterSlider.num_steps must be >= 2.")

        # log_scale
        self.log_scale = bool(self.log_scale)
        if self.log_scale and self.min_value <= 0:
            raise ValueError("When log_scale is True, min_value must be > 0.")
            
        # is_integer
        self.is_integer = bool(self.is_integer)

def create_interactive_vectorization_plot(vectorization_computation_function,
                                          slider_params: ParameterSlider,
                                          is_image=False,
                                          title: str = "Interactive Vectorization Visualization"):
    """
    Create an interactive Plotly figure with a slider to control vectorization computation.
    
    Parameters
    ----------
    vectorization_computation_function : callable
        Function that takes 1 parameter and returns numpy array of shape (N, M).
    slider_params : ParameterSlider
        Parameters for the slider.
    is_image : bool, optional
        If True, displays result as an image (heatmap). If False, displays each row as a line plot.
    title: str, optional
        Title of the interactive plot.
        
    Returns
    -------
    plotly.graph_objects.Figure
        Interactive Plotly figure with slider
    """
    
    def _create_slider_values(params: ParameterSlider):
        """Helper function to create slider values based on ParameterSlider configuration"""
        if params.log_scale:
            log_min = np.log10(params.min_value)
            log_max = np.log10(params.max_value)
            log_values = np.linspace(log_min, log_max, params.num_steps)
            values = 10**log_values
        else:
            values = np.linspace(params.min_value, params.max_value, params.num_steps)
        
        # Convert to appropriate type based on is_integer flag
        if params.is_integer or np.allclose(values, np.round(values)):
            values = np.round(values).astype(int)
        
        # Remove duplicates while preserving order
        if len(values) > len(set(values)):
            _, unique_indices = np.unique(values, return_index=True)
            values = values[np.sort(unique_indices)]
        
        return values
    
    # Create slider values
    slider_values = _create_slider_values(slider_params)
    
    # Create figure
    fig = go.Figure()
    
    # Store all traces for different parameter values
    trace_index = 0
    trace_map = {}  # maps slider_idx → list of trace indices to show
    
    # Define colors for different rows/lines
    colors = ['blue', 'red', 'green', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive', 'cyan']
    
    # Generate data for all parameter values
    for i, val in enumerate(slider_values):
        current_visible_traces = []
        
        # Call the vectorization function
        result = vectorization_computation_function(val)
        # Ensure result is numpy array
        result = np.array(result)
        
        if len(result.shape) != 2:
            raise ValueError("The vectorization_computation_function must return a 2D numpy array. If you are returning a 1D array, reshape it to (1, N).")
        
        if is_image:
            # Display as heatmap/image
            fig.add_trace(go.Heatmap(
                z=result,
                colorscale='viridis',
                visible=(i == 0),
                showscale=(i == 0),
                name=f"Result ({val})"
            ))
            current_visible_traces.append(trace_index)
            trace_index += 1
        else:
            # Display each row as a line
            for row_idx, row in enumerate(result):
                fig.add_trace(go.Scatter(
                    y=row,
                    x=np.arange(len(row)),
                    mode='lines',
                    name=f'Row {row_idx + 1}',
                    line=dict(color=colors[row_idx % len(colors)], width=2),
                    visible=(i == 0),
                    showlegend=(i == 0)  # Only show legend for first parameter value
                ))
                current_visible_traces.append(trace_index)
                trace_index += 1
        
        # Store trace indices for this parameter value
        trace_map[i] = current_visible_traces
    
    # Create slider steps
    total_traces = trace_index
    steps = []
    
    for i, val in enumerate(slider_values):
        visible = [False] * total_traces
        
        # Enable traces for current slider step
        if i in trace_map:
            for idx in trace_map[i]:
                visible[idx] = True
        
        title_text = f"{title} - {slider_params.name}: {val}"
        steps.append(dict(
            method="update",
            args=[{"visible": visible},
                  {"title": title_text}],
            label=f"{val}"
        ))
    
    # Add slider
    fig.update_layout(
        sliders=[dict(
            steps=steps,
            currentvalue={"prefix": f"{slider_params.name}: "},
            active=0,
            len=0.8,
            x=0.1,
            y=0,
            xanchor="left",
            yanchor="top"
        )]
    )
    
    # Update layout
    display_type = "Heatmap" if is_image else "Line Plot"
    initial_title = f"{title} - {display_type} - {slider_params.name}: {slider_values[0]}"
    
    fig.update_layout(
        height=600,
        showlegend=not is_image,
        title=initial_title,
        margin=dict(b=100)  # Bottom margin for slider
    )
    
    # Update axes labels
    if is_image:
        fig.update_xaxes(title_text="Column Index")
        fig.update_yaxes(title_text="Row Index")
    else:
        fig.update_xaxes(title_text="Index")
        fig.update_yaxes(title_text="Value")
    
    # Store trace map and slider values in figure for potential future use
    fig._trace_map = trace_map
    fig._slider_values = slider_values
    
    fig.show()
    return fig


**WARNING Depending on your notebook renderer, you might need to switch from `inline` to `notebook`, if when trying to render a plot nothing shows up.**

In [ ]:
%matplotlib inline
#%matplotlib notebook

# Data Exploration

This section shows you a detailed path for analyzing the data and explore which filtrations or data transformations would be better to discriminate between the three classes. Any filtration can be studied through the bottleneck distance matrix, which gives you an idea of how well your inputs are separated with respect to their class.

### Question 1. 
Plot a few of the time series to get an idea of the corresponding point clouds in $\mathbb{R}^3$.

In [ ]:
data_A_sample = data_A[1]
plot_3d_point_series({"data_A_sample": data_A_sample}, show_lines=True)

### Question 2. 
Compute and plot the persistence diagrams of the [alpha-complex filtration](https://gudhi.inria.fr/doc/3.7.0/group__alpha__complex.html), for a few examples of the time series.

In [ ]:
data_ABC = data
nb_data = np.shape(data_ABC)[0]

#Compute all persistence diagrams
diag_list = []
diag_list_dim0 = []
diag_list_dim1 = []
for i in range(nb_data):
    pt_cloud = data_ABC[i,:,:]
    alpha_complex = gd.AlphaComplex(points=pt_cloud)
    st = alpha_complex.create_simplex_tree()
    diag = st.persistence()
    diag_list.append(diag)
    diag_list_dim0.append(st.persistence_intervals_in_dimension(0))
    diag_list_dim1.append(st.persistence_intervals_in_dimension(1))
    if i%30 == 0:
        print('number of computed diagrams:', i)

In [ ]:
gd.plot_persistence_diagram(diag_list[25])

### Question 3.
Based on the persistence diagrams, compute the matrix of pairwise distances between these diagrams. You can use plotly to visualize a small number of distances (up to 10 compared diagrams) or imshow to see the full matrix.


In [ ]:
B0 = compute_distance_matrix(diag_list_dim0, distance_type='bottleneck')
B1 = compute_distance_matrix(diag_list_dim1, distance_type='bottleneck')

In [ ]:
#Save the pairwise distance matrices
pickle.dump(B0, open("B0.dat", "wb"))
pickle.dump(B1, open("B1.dat", "wb"))

In [ ]:
#To load the pairwise distance matrices 
B0 = pickle.load(open("B0.dat", "rb"), encoding="latin1")
B1 = pickle.load(open("B1.dat", "rb"), encoding="latin1")

In [ ]:
# Show the full matrix
plt.figure()
plt.imshow(B1, cmap='gray', interpolation='nearest');
plt.show()

In [ ]:
# See a detailed small version of the matrix
display_index = np.arange(0, 240, 20)
display_mat = B0[np.ix_(display_index, display_index)]
plot_distance_matrices({"Bottleneck (0h order homology)": display_mat}, diag_names=[train_labels[i] + "_" + str(i) for i in display_index])

In [ ]:
# See a detailed small version of the matrix
display_index = np.arange(0, 240, 20)
display_mat = B1[np.ix_(display_index, display_index)]
plot_distance_matrices({"Bottleneck (1st order homology)": display_mat}, diag_names=[train_labels[i] + "_" + str(i) for i in display_index])

In [ ]:
# Plot 2D MDS for dimension 0 persistence diagrams
pos0 = plot_mds(B0, train_labels, visualization_dim=2,
                    title="MDS 2D - Dimension 0 Persistence Diagrams",
                    color_map={'A': 'red', 'B': 'green', 'C': 'blue'})

# Plot 3D MDS for dimension 1 persistence diagrams
pos1 = plot_mds(B1, train_labels, visualization_dim=3,
                    title="MDS 3D - Dimension 1 Persistence Diagrams",
                    color_map={'A': 'red', 'B': 'green', 'C': 'blue'})

### Question 4.
We would like to compare the previous filtration with another one based on Rips complexes.

Use the Gudhi utility TimeDelayEmbedding (see Point Cloud Utilities manual https://gudhi.inria.fr/python/latest/point_cloud.html) to embed the data in dimension 3×3 = 9 with a delay equal to 2 (time-delay embedding) and do the same experiments as previously, using the Vietoris-Rips filtration this time.

In [ ]:
from gudhi.point_cloud.timedelay import TimeDelayEmbedding

edim=3
delay=2
diag_list = []
diag_list_TD_dim0 = []
diag_list_TD_dim1 = []

for i in range(nb_data):
    pt_cloud = TimeDelayEmbedding(dim=9,delay=1, skip=1)(data_ABC[i,:,:])
    rips_complex = gd.RipsComplex(points=pt_cloud,max_edge_length=100.0)
    st = rips_complex.create_simplex_tree(max_dimension=2)
    diag_list.append(st.persistence())
    diag_list_TD_dim0.append(st.persistence_intervals_in_dimension(0))
    diag_list_TD_dim1.append(st.persistence_intervals_in_dimension(1))
    if i%10 == 0:
        print('number of computed diagrams:', i)

In [ ]:
# Save persistence diagrams to not recompute them every time !
pickle.dump(diag_list, open("Diags_accelero.dat", "wb"))
pickle.dump(diag_list_TD_dim0, open("Diags_accelero_TD0.dat", "wb"))
pickle.dump(diag_list_TD_dim1, open("Diags_accelero_TD1.dat", "wb"))

In [ ]:
B_TD_0= compute_distance_matrix(diag_list_TD_dim0, distance_type='bottleneck')
B_TD_1= compute_distance_matrix(diag_list_TD_dim1, distance_type='bottleneck')

In [ ]:
pickle.dump(B_TD_0, open("B_TD_0.dat", "wb")) 
pickle.dump(B_TD_1, open("B_TD_1.dat", "wb"))

In [ ]:
#To load pairwise distance matrices 
B_TD_0 = pickle.load(open("B_TD_0.dat", "rb"), encoding="latin1") 
B_TD_1 = pickle.load(open("B_TD_1.dat", "rb"), encoding="latin1")

In [ ]:
# Plot 2D MDS for dimension 0 persistence diagrams
pos0 = plot_mds(B_TD_0, train_labels, visualization_dim=2,
                    title="MDS 2D - Dimension 0 Persistence Diagrams (Time delay)",
                    color_map={'A': 'red', 'B': 'green', 'C': 'blue'})

# Plot 3D MDS for dimension 1 persistence diagrams
pos1 = plot_mds(B_TD_1, train_labels, visualization_dim=3,
                    title="MDS 3D - Dimension 1 Persistence Diagrams (Time delay)",
                    color_map={'A': 'red', 'B': 'green', 'C': 'blue'})

# Vectorization

### Question 5.

Here is an example of exploring a hyperparameter on the landscape computation using `create_interactive_vectorization_plot`. First you define a function that takes as input your parameter and returns the vectorization associated to the input parameter. You pass that function to `create_interactive_vectorization_plot` and you specify the range of theslider to be displayed. Try changing the below function to modify either the vectorization method and the parameter driven by the slider.

In [ ]:
diag_list=pickle.load(open("Diags_accelero.dat", "rb"))
diag_list0=pickle.load(open("Diags_accelero_TD0.dat","rb"))
diag_list1=pickle.load(open("Diags_accelero_TD1.dat","rb"))

In [ ]:
selected_diag = diag_list1[0]

def compute_landscape(resolution):
    num_landscapes = 5
    filtered_diag = gdr.DiagramSelector(use=True, point_type="finite").fit_transform([selected_diag])[0]
    L0=gdr.Landscape(num_landscapes=num_landscapes, resolution=resolution, sample_range=[0.0,1.0]).fit_transform([filtered_diag])
    L0 = L0.reshape((num_landscapes, -1))
    return L0

# Define the parameter for the slider
slider_parameter = ParameterSlider(
    name="Resolution",
    min_value=10,
    max_value=1000,
    num_steps=20,
    log_scale=True,
    is_integer=True
)

# Plot the interactive landscape with resolution as parameter
fig = create_interactive_vectorization_plot(
    compute_landscape,
    slider_parameter,
    is_image=False,
    title="Landscape for 1st Homology"
)

Try using other vectorization methods and explore which ones seem to provide the best representation for classification. Recall that the output of your vectorization will be the input of your classifier.

# Classification

### Question 6.

Based on scikit learn classifiers (or any other library if you feel confortable with it), write a training pipeline that takes as input the dataset of diagrams you generated from the filtration you chose, cleans it, vectorizes it and then passes it to the classifier of your choice. You can do it for multiple pipelines and compare them in order to select the ones that performs the best. In the end, it is expected to have some basic analysis of why your model works (or not).

### Question 7.

Using the best model you have obtained, compute the estimated labels of `eval_data` and print them in a readable way. Provide also a summary of the number of each elements that fall into each class.

In [ ]:
# Perform the evaluation
# Replace with your model prediction
eval_labels = ['ABC'[int(np.random.rand()*3)] for _ in range(len(eval_data))]

for i, label in enumerate(eval_labels):
    print(i, ":", label)
print("Total A count:", eval_labels.count('A'))
print("Total B count:", eval_labels.count('B'))
print("Total C count:", eval_labels.count('C'))